# 📦 00 — Préprocessing Unifié
## Chargement, Correction COVID, Feature Engineering, Split, Métriques

Ce notebook centralise **tout le préprocessing** pour les modèles de forecasting.
Les notebooks modèles (`01_SARIMA`, `02_HoltWinters`, etc.) importeront les données préparées.

### 📋 Étapes
1. Installation & Imports
2. Chargement & Agrégation mensuelle
3. Correction COVID (STL Interpolation)
4. Feature Engineering (lags, rolling, yoy, time features)
5. Split Train/Test + Walk-Forward Validation
6. Fonctions de métriques (MAPE, sMAPE, MASE, Diebold-Mariano)
7. Export des données préparées en CSV

## 1. 📦 Installation & Imports

In [ ]:
# Installation des dépendances (à lancer une seule fois)
import subprocess, sys

packages = [
    "prophet", "statsmodels", "scikit-learn",
    "psycopg2-binary", "matplotlib", "seaborn",
    "pandas", "numpy", "xgboost", "scipy"
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("✅ Toutes les dépendances sont installées")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime

# Stats
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.model_selection import TimeSeriesSplit

# ML
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Style
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams['figure.dpi'] = 120

# Paths
DATA_PATH = "../daily_revenue.csv"
OUTPUT_PATH = "prepared_data.csv"

print("✅ Imports OK")
print(f"   Pandas : {pd.__version__}")
print(f"   Numpy  : {np.__version__}")

## 2. 🗄️ Chargement & Agrégation Mensuelle

In [ ]:
# ─── Chargement du CSV journalier et agrégation mensuelle ───
df_daily = pd.read_csv(DATA_PATH, parse_dates=["date"])
df_daily = df_daily.set_index("date").sort_index()

df_monthly = df_daily["revenue"].resample("MS").sum().to_frame()

print(f"✅ Données chargées : {len(df_daily)} jours")
print(f"   Période : {df_daily.index.min().strftime('%Y-%m-%d')} → {df_daily.index.max().strftime('%Y-%m-%d')}")
print(f"✅ Agrégées mensuellement : {len(df_monthly)} mois")
print(f"   Revenue min  : {df_monthly['revenue'].min():>15,.0f} €")
print(f"   Revenue max  : {df_monthly['revenue'].max():>15,.0f} €")
print(f"   Revenue mean : {df_monthly['revenue'].mean():>15,.0f} €")
df_monthly.head(10)

## 3. 🔧 Correction COVID — STL Interpolation

In [ ]:
def correct_covid_stl(df, df_original, random_seed=42):
    """
    Corrige les mois COVID (Mars 2020 - Juin 2021) via STL :
    Décompose → remplace résidus anormaux → reconstruit
    """
    rng = np.random.default_rng(random_seed)
    df = df.copy()

    stl = STL(df["revenue"], period=12, robust=True)
    result = stl.fit()

    trend = result.trend
    seasonal = result.seasonal
    resid = result.resid

    covid_mask = ((df.index.year == 2020) & (df.index.month >= 3)) | \
                 ((df.index.year == 2021) & (df.index.month <= 6))

    resid_healthy = resid[~covid_mask].dropna()
    resid_std = resid_healthy.std()
    resid_mean = resid_healthy.mean()

    n_covid = covid_mask.sum()
    new_resid = resid.copy()
    new_resid[covid_mask] = rng.normal(resid_mean, resid_std, n_covid)

    df["revenue"] = (trend + seasonal + new_resid).clip(lower=0)
    corrections = df[covid_mask].index

    return df, corrections, result, covid_mask


print("🔧 Correction COVID — STL Interpolation")
print("=" * 50)
df_original = df_monthly.copy()
df_monthly, corrections_covid, stl_result, covid_mask = correct_covid_stl(df_monthly, df_original)

for year in [2019, 2020, 2021]:
    before = df_original.loc[df_original.index.year == year, "revenue"].sum()
    after = df_monthly.loc[df_monthly.index.year == year, "revenue"].sum()
    print(f"   {year}: {before/1e6:.2f}M€ → {after/1e6:.2f}M€")
print(f"✅ {len(corrections_covid)} mois COVID corrigés")

## 4. 🧠 Feature Engineering Avancé

In [ ]:
def add_covid_flags(df):
    df = df.copy()
    df["covid_severe"] = ((df.index.year == 2020) & (df.index.month.isin([3,4,5,6]))).astype(int)
    df["covid_moderate"] = (
        ((df.index.year == 2020) & (df.index.month.isin([7,8,9,10,11,12]))) |
        ((df.index.year == 2021) & (df.index.month.isin([1,2,3,4,5,6])))
    ).astype(int)
    df["covid_flag"] = (df["covid_severe"] | df["covid_moderate"]).astype(int)
    return df

def add_time_features(df):
    df = df.copy()
    idx = df.index
    df["month_sin"] = np.sin(2 * np.pi * idx.month / 12)
    df["month_cos"] = np.cos(2 * np.pi * idx.month / 12)
    df["quarter_sin"] = np.sin(2 * np.pi * idx.quarter / 4)
    df["quarter_cos"] = np.cos(2 * np.pi * idx.quarter / 4)
    df["is_december"] = (idx.month == 12).astype(int)
    df["is_summer"] = idx.month.isin([6,7,8]).astype(int)
    df["is_january"] = (idx.month == 1).astype(int)
    t = np.arange(len(df))
    df["trend"] = t
    df["trend_sq"] = t ** 2
    return df

def add_lag_features(df, lags=[1,2,3,6,12]):
    df = df.copy()
    for lag in lags:
        df[f"lag_{lag}"] = df["revenue"].shift(lag)
    return df

def add_rolling_features(df):
    df = df.copy()
    df["rolling_3"] = df["revenue"].shift(1).rolling(3).mean()
    df["rolling_6"] = df["revenue"].shift(1).rolling(6).mean()
    df["rolling_12"] = df["revenue"].shift(1).rolling(12).mean()
    df["volatility_3"] = df["revenue"].shift(1).rolling(3).std()
    df["volatility_6"] = df["revenue"].shift(1).rolling(6).std()
    df["rolling_min_6"] = df["revenue"].shift(1).rolling(6).min()
    df["rolling_max_6"] = df["revenue"].shift(1).rolling(6).max()
    return df

def add_yoy_features(df):
    df = df.copy()
    df["yoy_growth"] = df["revenue"].pct_change(periods=12).clip(-50, 50) * 100
    df["yoy_ratio"] = (df["revenue"] / df["revenue"].shift(12)).clip(0.5, 2.0)
    return df

In [ ]:
# Appliquer toutes les transformations
df_feat = add_covid_flags(df_monthly)
df_feat = add_time_features(df_feat)
df_feat = add_lag_features(df_feat)
df_feat = add_rolling_features(df_feat)
df_feat = add_yoy_features(df_feat)

# Supprimer les NaN (début de série sans lags)
df_clean = df_feat.dropna()
nan_count = len(df_feat) - len(df_clean)

print(f"✅ Feature Engineering terminé")
print(f"   Dimensions : {df_clean.shape[0]} lignes × {df_clean.shape[1]} colonnes")
print(f"   NaN supprimés : {nan_count}")
print(f"\n📊 Features ({df_clean.shape[1]} totales) :")
for col in df_clean.columns:
    print(f"   • {col}")

# Matrice de corrélation
fig, ax = plt.subplots(figsize=(14, 10))
corr = df_clean.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0, linewidths=0.5, ax=ax)
ax.set_title("Matrice de Corrélation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("feature_correlation.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\n🔗 Top corrélations avec 'revenue' :")
corr_target = corr["revenue"].drop("revenue").sort_values(key=abs, ascending=False)
for feat, val in corr_target.head(10).items():
    print(f"   {feat:>15} : {val:+.3f}")

## 5. ✂️ Split Train/Test + Walk-Forward Validation

In [ ]:
FEATURES = [c for c in df_clean.columns if c != "revenue"]
TARGET = "revenue"

train = df_clean[df_clean.index.year < 2022].copy()
test  = df_clean[df_clean.index.year == 2022].copy()

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f"📊 Split Train/Test")
print(f"   Train : {len(train)} mois ({train.index.min().strftime('%b %Y')} → {train.index.max().strftime('%b %Y')})")
print(f"   Test  : {len(test)} mois  ({test.index.min().strftime('%b %Y')} → {test.index.max().strftime('%b %Y')})")
print(f"   Features : {len(FEATURES)}")
print(f"   X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"   X_test : {X_test.shape}, y_test : {y_test.shape}")

## 6. 📐 Fonctions de Métriques

In [ ]:
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask = denom != 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def mase(y_true, y_pred, y_train):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    y_train = np.array(y_train, dtype=float)
    naive = np.mean(np.abs(np.diff(y_train, 12)))
    return np.nan if naive == 0 else np.mean(np.abs(y_true - y_pred)) / naive

def diebold_mariano(y_true, y_pred1, y_pred2):
    y_true, y_pred1, y_pred2 = np.array(y_true), np.array(y_pred1), np.array(y_pred2)
    d = np.abs(y_true - y_pred1) - np.abs(y_true - y_pred2)
    d = d[~np.isnan(d)]
    if len(d) < 2: return 1.0, 0.0
    mean_d, var_d = np.mean(d), np.var(d, ddof=1) / len(d)
    if var_d <= 0: return 1.0, 0.0
    dm_stat = mean_d / np.sqrt(var_d)
    return dm_stat, 2 * (1 - stats.norm.cdf(abs(dm_stat)))

def evaluate_model(name, y_true, y_pred, y_train=None, verbose=True):
    y_true, y_pred = np.array(y_true, dtype=float), np.array(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_val = mape(y_true, y_pred)
    smape_val = smape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    mase_val = mase(y_true, y_pred, y_train) if y_train is not None else np.nan
    
    if verbose:
        print(f"  {'─'*45}")
        print(f"  📊 {name}")
        print(f"     MAE   : {mae:>11,.0f} €")
        print(f"     RMSE  : {rmse:>11,.0f} €")
        print(f"     MAPE  : {mape_val:>10.2f} %")
        print(f"     sMAPE : {smape_val:>10.2f} %")
        print(f"     R²    : {r2:>10.4f}")
        if not np.isnan(mase_val):
            print(f"     MASE  : {mase_val:>10.3f}  {'✅ Bat la naïveté' if mase_val < 1 else '⚠️ Ne bat pas'}")
    
    return {"Modèle": name, "MAE (€)": mae, "RMSE (€)": rmse,
            "MAPE (%)": mape_val, "sMAPE (%)": smape_val, "R²": r2, "MASE": mase_val}

print("✅ Fonctions de métriques prêtes")

## 7. 💾 Export des Données Préparées

In [ ]:
# Sauvegarder les données préparées
df_clean.to_csv("prepared_data.csv")
print(f"💾 Données sauvegardées → prepared_data.csv ({len(df_clean)} lignes)")

# Sauvegarder aussi les splits pour les modèles ML
train.to_csv("train_data.csv")
test.to_csv("test_data.csv")
print(f"💾 Train → train_data.csv ({len(train)} lignes)")
print(f"💾 Test  → test_data.csv ({len(test)} lignes)")

print("\n✅ Préprocessing terminé !")
print("   Les notebooks modèles peuvent maintenant importer ces CSV.")